## Stock Price Predictor

In [ ]:
import numpy as np
import pandas as pd

import yfinance as yf

import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
stock = input('Enter the ticker symbol (e.g. GC=F, AAPL, TCS.NS): ').strip().upper() or 'GC=F'
data = yf.download(stock, period='10y', auto_adjust=True, progress=False)
if isinstance(data.columns, pd.MultiIndex):      # newer yfinance returns MultiIndex columns
    data.columns = data.columns.get_level_values(0)
data = data[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.Close.plot(figsize=(10,7), color='r')
plt.xlabel(f"{stock} Prices")
plt.ylabel(f"{stock} Prices Series")
plt.show()

In [ ]:
sns.histplot(data.Open, kde=True)

In [ ]:
sns.histplot(data.High, kde=True)

In [ ]:
sns.histplot(data.Low, kde=True)

In [ ]:
sns.histplot(data.Close, kde=True)

### Conlusions
- Shape the of the data
- How our data is distributed
- It's is very very non-linear 

In [ ]:
X = data.drop('Close', axis = 1)
y = data['Close']

## Training and Testing Dataset

In [ ]:
# Chronological split: train on the past, test on the most recent 20% (a random split leaks future data)
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [ ]:
print(X_train.shape, X_test.shape,y_train.shape,y_test.shape)

## Modeling Part

In [ ]:
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
pred = model_lr.predict(X_test)

In [ ]:
# Predict value print
pred

In [ ]:
# MSE (Mean Square Error)
# RMSE (Root Mean Square Error)
# r2_square --> 1.0 -> Very Good Model

from sklearn.metrics import mean_squared_error, r2_score

def calculate_metrics(y_test, y_pred):
    '''
        y_test: ground truth
        y_pred: model predict value
    '''
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2_scores = r2_score(y_test, y_pred)
    print('*'*50)
    print(f"*  Mean Square Error: {mse}")
    print(f"*  Root Mean Square Error: {rmse}")
    print(f"*  r2_score: {r2_scores}")
    print('*'*50)

calculate_metrics(y_test, pred)

### Lasso and Ridge Model

In [ ]:
from sklearn.linear_model import Lasso, Ridge

model_lasso = Lasso().fit(X_train, y_train)
model_ridge = Ridge().fit(X_train, y_train)

model_lasso_pred = model_lasso.predict(X_test)
model_ridge_pred = model_ridge.predict(X_test)

In [ ]:
calculate_metrics(y_test, model_lasso_pred)

calculate_metrics(y_test, model_ridge_pred)

## Support Vector Machine Regression

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.1, 1, 10, 100, 1000],
    'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
    'kernel': ['rbf']
}

params_grid = GridSearchCV(SVR(), params, refit=True, verbose=3)
params_grid.fit(X_train, y_train)

In [ ]:
model_svr = SVR(C=10, gamma=0.1, kernel='rbf')
model_svr.fit(X_train, y_train)
svr_pred = model_svr.predict(X_test)

## Model Save

In [ ]:
import joblib

joblib.dump(model_ridge, 'model.pkl')
model = joblib.load('model.pkl')